<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Decisions and repetition

**[C02-M02-L03 · Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/)** · C02, Python for Practical AI Engineering · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** make decisions with `if`, `elif` and `else`, use loops to filter tickets and count them by category, stop a loop early, write a list comprehension, and give a correct answer when there are no tickets.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | Nothing to install, and no internet connection: only Python. |
| **You should know** | How to read values from a list of dictionaries, and the basic `for` loop, from [Collections](https://learning.nextia-ai.com/courses/python/m02/collections/). |
| **You do not need** | The local project from Module 1. The setup makes a `ticket-cleaner` folder here, for the script of this lesson. |
| **Tested** | 2026-10-07, Python 3.14.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M02-L03-decisions-and-repetition/decisions-and-repetition-solution.ipynb).

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It is empty except for the `data` folder, as at the end of [Files and terminal](https://learning.nextia-ai.com/courses/python/m01/files-and-terminal/#guided-practice-make-your-project-folder). It needs no internet connection.

You should see `Ready: ticket-cleaner`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)


os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

import re
from pathlib import Path

def check_count_tickets():
    """Run count_tickets.py for 'open' and 'resolved', and compare the output."""
    script = Path("count_tickets.py").read_text(encoding="utf-8")
    cases = {
        "open": "Tickets with status 'open': 3\n  billing: 2\n  login: 1\nAverage priority: 2.0\n",
        "resolved": "Tickets with status 'resolved': 0\nNo tickets to count.\n",
    }
    for status, expected in cases.items():
        variant = re.sub(r"^wanted_status = .*$", f'wanted_status = "{status}"', script, count=1, flags=re.M)
        result = subprocess.run([sys.executable, "-c", variant], capture_output=True, text=True)
        if result.returncode:
            print(f"Not yet: for '{status}', the script stops with {result.stderr.strip().splitlines()[-1]}")
            return
        if result.stdout != expected:
            print(f"Not yet: for '{status}', your output is:\n{result.stdout}\nThe task expects:\n{expected}")
            return
    print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Decisions: if, elif and else

An `if` statement runs some lines only when a condition is `True`. Python checks the conditions from the top, and runs the block under the **first** condition that is `True`. `elif` means "else, if". The `else` block runs only when no condition was `True`. Each block is indented by four spaces.

> **Predict.** Which label does each priority get? Then run the cell.

In [ ]:
for priority in [1, 2, 3]:
    if priority == 1:
        label = "urgent"
    elif priority == 2:
        label = "normal"
    else:
        label = "low"
    print(priority, label)

> **Check.** You should see `1 urgent`, `2 normal` and `3 low`.

A condition does not have to be a comparison. Python treats `0`, `""`, `None` and an empty list, dictionary or set as false. So `if not selected:` means "if `selected` is empty". Run the cell.

In [ ]:
for value in [0, 3, "", "open", None, [], ["T-1001"]]:
    if value:
        print(repr(value), "counts as true")
    else:
        print(repr(value), "counts as false")

> **Check.** You should see that `3`, `'open'` and `['T-1001']` count as true, and the other four count as false.

## 2. A loop with a result

A loop that builds a result has three parts: an empty result before the loop, add to it in the loop, and use it after the loop. Run the cell. It makes the six tickets of this module, and keeps the open ones.

In [ ]:
tickets = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]

selected = []                          # 1. an empty result
for ticket in tickets:
    if ticket["status"] == "open":
        selected.append(ticket)        # 2. add to it
print(len(selected))                   # 3. use it

> **Check.** You should see `3`.

## 3. Worked example: count tickets by category

The result is a dictionary: category → count. `category_counts.get(category, 0)` gives the count so far, or `0` the first time that a category appears. The `print` in the loop shows the state after each step: a **loop trace**. Run the cell.

In [ ]:
category_counts = {}
for ticket in tickets:
    category = ticket["category"]
    category_counts[category] = category_counts.get(category, 0) + 1
    print(ticket["id"], category, category_counts)

print("Result:", category_counts)

> **Check.** The last line is `Result: {'billing': 3, 'login': 2, 'shipping': 1}`. In steps 1 and 6, the category is new, so `.get()` gives `0`.

> **Your turn: change one thing.** In the cell above, count by `"status"` instead of `"category"`. Predict the result first. (You should see `{'open': 3, 'closed': 2, 'pending': 1}`.)

## 4. Stop early, or skip one step

`break` ends the loop at once. `continue` ends this step only, and goes to the next item. This loop finds the first open ticket with priority 1. `first_urgent` starts as `None`, which after the loop means "not found".

> **Predict.** How many `checking` lines do you expect? Then run the cell.

In [ ]:
first_urgent = None
for ticket in tickets:
    print("checking", ticket["id"])
    if ticket["status"] == "open" and ticket["priority"] == 1:
        first_urgent = ticket
        break

if first_urgent is None:
    print("No urgent open ticket.")
else:
    print("First urgent open ticket:", first_urgent["id"])

> **Check.** You should see 3 `checking` lines, then `First urgent open ticket: T-1003`.

> **Your turn: change one thing.** Remove the `break` line, and run the cell again. What changes, and is the answer still correct? (All six tickets are checked. The answer is the same here, because no later ticket matches. With more data, the loop would keep the **last** match, not the first.)

## 5. List comprehensions: a short form

A **list comprehension** is a short filter: **what** to put in the new list, **for** each item, **if** which condition. Use it when it fits on one line and reads clearly. Run the cell.

In [ ]:
open_ids = [ticket["id"] for ticket in tickets if ticket["status"] == "open"]
print(open_ids)

> **Check.** You should see `['T-1001', 'T-1003', 'T-1005']`.

## 6. Guided practice: build count_tickets.py

Now you write the script of the lesson, as a file in the project folder. `%%writefile count_tickets.py` on the first line of a cell saves the rest of the cell as that file. Run the cell: you see `Writing count_tickets.py` (or `Overwriting`, the second time).

`sorted()` puts the categories in alphabetical order, so the output is the same each time.

In [ ]:
%%writefile count_tickets.py
tickets = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]

wanted_status = "open"

# Keep only the tickets with the wanted status.
selected = []
for ticket in tickets:
    if ticket["status"] == wanted_status:
        selected.append(ticket)

# Count the selected tickets in each category.
category_counts = {}
for ticket in selected:
    category = ticket["category"]
    category_counts[category] = category_counts.get(category, 0) + 1

print(f"Tickets with status '{wanted_status}': {len(selected)}")
for category, count in sorted(category_counts.items()):
    print(f"  {category}: {count}")

Run the script, as `python count_tickets.py` does in your terminal.

In [ ]:
run("count_tickets.py")

> **Check.** You should see `Tickets with status 'open': 3`, `  billing: 2`, `  login: 1` and `(exit code 0)`.

## 7. Your turn: no matching tickets

The next cell is the same script, with `wanted_status = "resolved"`. No ticket has that status.

> **Predict.** What does the script print? Run the cell, then the `run` cell after it.

The script prints `Tickets with status 'resolved': 0` and nothing else. That is correct, but a reader cannot tell if the script worked or stopped early.

> **Your turn.** Change the report part of the script (the lines from `print(f"Tickets with ...`), in two steps:
>
> 1. Add one decision, so that for `"resolved"` the output is `Tickets with status 'resolved': 0` and then `No tickets to count.`
> 2. Add one more line to the report: `Average priority: ` and the average priority of the selected tickets, rounded to 1 decimal place. For `"open"`, it is `2.0`. For `"resolved"`, the script must not stop with an error.
>
> Run the cell and the `run` cell. Then run the check cell: it runs your script for `"open"` and for `"resolved"`.

In [ ]:
%%writefile count_tickets.py
tickets = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": 2},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": 3},
    {"id": "T-1003", "status": "open", "category": "login", "priority": 1},
    {"id": "T-1004", "status": "pending", "category": "billing", "priority": 2},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": 3},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": 1},
]

wanted_status = "resolved"

# Keep only the tickets with the wanted status.
selected = []
for ticket in tickets:
    if ticket["status"] == wanted_status:
        selected.append(ticket)

# Count the selected tickets in each category.
category_counts = {}
for ticket in selected:
    category = ticket["category"]
    category_counts[category] = category_counts.get(category, 0) + 1

print(f"Tickets with status '{wanted_status}': {len(selected)}")
for category, count in sorted(category_counts.items()):
    print(f"  {category}: {count}")

In [ ]:
run("count_tickets.py")

In [ ]:
check_count_tickets()

## 8. Failure case: ZeroDivisionError on empty input

Tomás calculates the average without a decision first. It works for `"open"`. Here is the same calculation for a filter that matches nothing.

> **Predict.** What happens? Then run the cell. (The `try` and `except` lines catch the error, so that the notebook continues. Module 5 explains them.)

In [ ]:
selected = []
total_priority = 0
try:
    average_priority = total_priority / len(selected)
except ZeroDivisionError as error:
    print("ZeroDivisionError:", error)

> **Check.** You should see `ZeroDivisionError: division by zero`. Empty input is a normal case: a new day with no tickets gives an empty list, and so does a filter with a typing mistake.

**Fix:** decide what the correct answer is for empty input. Check for it **before** the division, with `if not selected:`. Test the script with input that matches nothing, every time that you change it.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/). They count toward your certificate when you are signed in and enrolled. For the module check, set `wanted_status` to `"closed"` in your script, and predict the output before you run it.

In this lesson you learned that `if`, `elif` and `else` run the block of the first true condition, that a loop builds a result in three parts, that `break` and `continue` change how a loop runs, and that a script must give a correct answer for empty input.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.